# DINO notebook with U-Net refinement

This notebook segments structures in Imaris microscopy images with a pretrained DINOv2 backbone, and then tests whether a U-Net can refine the result.

The DINO pipeline:

- A pretrained DINOv2 backbone extracts feature embeddings for all image patches.
- Positive points inside the object and negative points in the background are sampled for each image.
- The patches containing those points form a positive and a negative prototype set. The cosine similarity between every image patch and the prototypes gives a similarity heatmap.
- Thresholding the heatmap gives the segmentation mask.

**Supervision.** The DINO backbone is self-supervised and is never trained on our data. The reference points, however, are sampled from the expert ground-truth masks, so they act as simulated user clicks. The DINO step is therefore best described as few-shot, or interactive, segmentation with a frozen self-supervised backbone, not as a fully unsupervised method.

**U-Net refinement.** A U-Net is fed with the raw image, the DINO heatmap, the raw image and the heatmap, or the raw image, the heatmap and the DINO mask. It is trained against the expert ground-truth masks, so this step is fully supervised and should be viewed as an addition to the pipeline, not a part of it.

**Evaluation protocol.**

- Every file contributes 100 evenly spaced z-slices. Every fourth slice is set aside as a *tuning set*. All hyperparameters of the DINO step are chosen on the tuning set only, and every reported DINO result uses the remaining *evaluation set*. Tuning and evaluation slices are interleaved, so neighbouring slices are correlated and some optimism remains.
- The U-Net uses leave-one-sample-out cross-validation. Model selection uses validation slices from the training samples, and the held-out sample is used only for the final test.
- Slices from the same sample are correlated, and so are seeds on the same slice. Results are therefore averaged over seeds per slice, then over slices per sample, and confidence intervals are computed across samples.

The overall goal is to investigate how well the DINO approach segments the images, and whether the supervised U-Net addition improves the segmentation quality.

# 1. Setup

In [ ]:
# Pinned versions so that the notebook installs the same packages every time
%pip install -q SciencePlots==2.2.2 imaris-ims-file-reader==0.1.8

import os
# Lets PyTorch reuse fragmented GPU memory; set before torch is imported
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc
import random
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.ndimage as ndi
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import scienceplots  # Registers the "science" matplotlib styles
from skimage.filters import threshold_otsu
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from imaris_ims_file_reader.ims import ims
from IPython.display import display

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Apple Silicon GPU, makes local runs on a Mac feasible for the smaller backbones
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device, "| torch", torch.__version__, "| numpy", np.__version__)

# DINOv2 code pinned to a fixed commit so torch.hub always loads the same model definition
DINOV2_REPO = "facebookresearch/dinov2:7764ea0f912e53c92e82eb78a2a1631e92725fc8"

# Data location. On Kaggle the dataset is found automatically under /kaggle/input, wherever it is mounted.
# Anywhere else, set the environment variable LIU_DATA_DIR or edit the fallback path below.
ON_KAGGLE = Path("/kaggle/input").exists()

def find_kaggle_data_dir():
    """Returns the folder under /kaggle/input that contains the .ims files"""
    hits = sorted(Path("/kaggle/input").rglob("*.ims")) if ON_KAGGLE else []
    return hits[0].parent if hits else None

DATA_DIR = Path(os.environ.get("LIU_DATA_DIR") or find_kaggle_data_dir() or "data/liudata")
# Results and checkpoints go to the output folder, which is what "kaggle kernels output" downloads
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULTS_DIR = OUTPUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    """Writes a result table as soon as it exists and prints it, so it survives a later failure and shows in the run log"""
    df.to_csv(RESULTS_DIR / f"{name}.csv", index=False)
    print(f"[{name}]\n{df.round(4).to_string()}\n", flush=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
# Slices read from the .ims files are cached here, which makes reruns much faster. On Kaggle the cache lives in /tmp,
# so the several GB of cached slices are not included in the downloadable output.
CACHE_DIR = Path(os.environ.get("LIU_CACHE_DIR", "/tmp/liu_cache" if ON_KAGGLE else "cache"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Data directory:", DATA_DIR)

# The ImageNet mean and standard deviation that Meta uses
imagenet_mn = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def seeding(seed):
    """Ensuring that the runs are random but reproducible"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

plt.style.use(["science", "grid", "no-latex"])

In [ ]:
@dataclass
class Config:
    seed: int = 9

    # Data selection
    z_margin: int = 40            # Slices skipped at the start and end of each stack
    slices_per_file: int = 100    # Evenly spaced z-slices loaded per file
    tuning_every: int = 4         # Every n-th slice goes to the tuning set
    clip_low: float = 1.0         # Intensity percentiles used to clip outliers.
    clip_high: float = 99.5       # 95 saturated much of the bright foreground

    # DINO input size. The longest image side is resized to vit_eval_size, keeping the aspect ratio.
    # Has to be a multiple of 14, the patch size for v2. Performance peaked before 1400 in earlier tests.
    vit_eval_size: int = 980
    # Optional common pixel size in micrometres. When set, every image is resampled to this pixel size so that
    # one DINO patch covers the same tissue area in all samples. vit_eval_size then only caps the longest side.
    target_um_per_px: float | None = None
    # Which transformer block to read the features from, counted from the end (1 = last block)
    dino_layer_from_end: int = 1

    # Prototype heatmap and thresholding. neg_weight, dino_layer_from_end, threshold_mode and top_percent
    # are re-selected on the tuning set in section 3.2.
    neg_weight: float = 0.8              # Lambda in pos_sim - lambda*neg_sim
    score_norm: str = "robust"           # Standardize each slice's scores by median and MAD ("robust") or not ("none").
                                         # Leaves thresholds set on the same slice unchanged; keeps a carried-over
                                         # threshold meaningful across slices and files (the transfer experiments)
    n_points: int = 25                   # Reference points per class
    boundary_margin: int = 5             # Minimum distance in pixels between sampled points and the GT boundary
    threshold_mode: str = "top_percent"  # "top_percent" or "otsu"
    top_percent: float = 10
    seeds: tuple = (0, 1, 2)

    # U-Net
    unet_backbone: str = "dinov2_s14"    # Small backbone due to computational limits
    unet_size: int = 960                 # Must be divisible by 8 for the 4-level U-Net
    unet_features: tuple = (32, 64, 128, 256)
    unet_batch_size: int = 4
    unet_lr: float = 1e-3
    unet_epochs: int = 30
    unet_patience: int = 6
    unet_thresholds: tuple = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
    unet_seeds: tuple = (0,)
    unet_slices_per_file: int = 25       # Train, validation and test slices per file

CFG = Config()
seeding(CFG.seed)

# 2. Data preparation, prototype and evaluation functions

## 2.1 Data loading

In [ ]:
def normalize_img(img, low, high):
    """Clipping intensity outliers at the given percentiles and scaling to [0, 1]"""
    img = np.asarray(img, np.float32)
    lo, hi = np.percentile(img, [low, high])
    img = np.clip(img, lo, hi)
    return ((img - lo) / max(hi - lo, 1e-8)).astype(np.float32)

def pick_slices(z_min, z_max, n_slices):
    """Picks a number of evenly spread z-slices within a range"""
    return np.linspace(z_min, z_max, n_slices).round().astype(int).tolist()

def evenly_spaced(items, n):
    """Picks n evenly spaced items, so that subsets cover the whole z-range instead of only the top of the stack"""
    if n >= len(items):
        return list(items)
    idx = np.linspace(0, len(items) - 1, n).round().astype(int)
    return [items[i] for i in idx]

def load_file_cases(name, info, cfg):
    """Reads the selected z-slices and GT masks of one .ims file, using a disk cache for reruns"""
    cache_path = CACHE_DIR / f"{name}_c{info['img_channel']}_n{cfg.slices_per_file}_m{cfg.z_margin}_clip{cfg.clip_low}-{cfg.clip_high}.npz"
    if cache_path.exists():
        data = np.load(cache_path)
        imgs, gts, z_list, px_um = data["imgs"], data["gts"], data["z"].tolist(), tuple(data["px_um"])
    else:
        file = ims(str(DATA_DIR / info["filename"]))
        print(f"File {name} shape:", file.shape)
        n_z = file.shape[2]
        z_list = pick_slices(cfg.z_margin, n_z - 1 - cfg.z_margin, cfg.slices_per_file)
        # The Liu files have the gt mask as the last channel
        gt_channel = file.shape[1] - 1
        # Voxel size from the file metadata, ordered (z, y, x) and normally in micrometres
        px_um = tuple(float(r) for r in file.resolution[1:])
        imgs, gts = [], []
        for z in z_list:
            img = normalize_img(file[0, info["img_channel"], z], cfg.clip_low, cfg.clip_high)
            gt = np.asarray(file[0, gt_channel, z]) > 0
            if img.shape != gt.shape:
                # Avoiding scale mismatches between GT masks and images
                img = np.asarray(F.interpolate(torch.from_numpy(img)[None, None], gt.shape, mode="bilinear", align_corners=False)[0, 0])
            # float16 halves the memory use, the precision is plenty for images in [0, 1]
            imgs.append(img.astype(np.float16))
            gts.append(gt)
        imgs, gts = np.stack(imgs), np.stack(gts)
        np.savez(cache_path, imgs=imgs, gts=gts, z=np.asarray(z_list), px_um=np.asarray(px_um))
    cases = []
    for i, z in enumerate(z_list):
        if not gts[i].any():
            print(f"Skipping file {name}, z={z}: the GT mask is empty")
            continue
        cases.append({"file": name, "z": int(z), "slice_idx": i, "img": imgs[i], "gt": gts[i], "px_um": px_um})
    return cases

def control_img(img):
    """Making sure that the image is a float32 tensor with the shape [C, H, W]"""
    if isinstance(img, np.ndarray):
        img = torch.from_numpy(np.asarray(img, np.float32))
    img = img.float()
    if img.ndim == 2:
        img = img.unsqueeze(0)
    return img.contiguous()

## 2.2 Evaluation metrics, thresholding and point sampling

In [ ]:
def dice(pred, gt):
    """Dice score. Two empty masks count as a perfect match"""
    tp = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2 * tp / denom)

def iou(pred, gt):
    """Intersection over union, i.e. the Jaccard index. Two empty masks count as a perfect match"""
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return 1.0 if union == 0 else float(inter / union)

def hd95(pred, gt, spacing=(1.0, 1.0)):
    """95th percentile Hausdorff distance between the mask boundaries, in the units of spacing (micrometres).
    Returns NaN when one of the masks is empty, since the distance is undefined then."""
    if not pred.any() or not gt.any():
        return np.nan
    d_to_gt = ndi.distance_transform_edt(~gt, sampling=spacing)
    d_to_pred = ndi.distance_transform_edt(~pred, sampling=spacing)
    bound_pred = pred & ~ndi.binary_erosion(pred)
    bound_gt = gt & ~ndi.binary_erosion(gt)
    all_distances = np.concatenate([d_to_gt[bound_pred], d_to_pred[bound_gt]])
    return float(np.percentile(all_distances, 95))

def seg_metrics(pred, gt, px_um, with_hd95=True):
    """All segmentation metrics for one prediction"""
    out = {"Dice": dice(pred, gt), "IoU": iou(pred, gt)}
    if with_hd95:
        out["HD95_um"] = hd95(pred, gt, px_um)
    return out

def threshold_heatmap(heat, cfg, pos_pts=None, neg_pts=None):
    """Turns a heatmap into a binary mask.
    top_percent keeps the given share of pixels, which assumes a known foreground fraction.
    otsu picks the threshold from the heatmap histogram and makes no such assumption.
    clicks puts the threshold halfway between the heat at the positive and at the negative reference points
    (10th and 90th percentiles when there are more than five), as the boneseg app does. In the app this beat
    otsu and top_percent on the demo stack and on Liu file A."""
    if cfg.threshold_mode == "top_percent":
        thr = np.percentile(heat, 100 - cfg.top_percent)
    elif cfg.threshold_mode == "otsu":
        thr = threshold_otsu(heat)
    elif cfg.threshold_mode == "clicks":
        if pos_pts is None or neg_pts is None or not len(pos_pts) or not len(neg_pts):
            thr = threshold_otsu(heat)
        else:
            hp = np.array([heat[int(y), int(x)] for y, x in pos_pts])
            hn = np.array([heat[int(y), int(x)] for y, x in neg_pts])
            lo = np.quantile(hp, 0.1) if len(hp) > 5 else hp.min()
            hi = np.quantile(hn, 0.9) if len(hn) > 5 else hn.max()
            thr = (lo + hi) / 2
    else:
        raise ValueError(f"Unknown threshold mode: {cfg.threshold_mode}")
    return heat >= thr, float(thr)

def _sample_region(region, k, rng):
    y, x = np.nonzero(region)
    # Sampling without replacement whenever possible, so that the points are distinct
    idx = rng.choice(len(y), k, replace=len(y) < k)
    return [(int(y[i]), int(x[i])) for i in idx]

def sample_pos_points(gt, k_points, seed, margin):
    """Sampling k points inside the GT mask, at least margin pixels from the boundary when the mask allows it"""
    if not gt.any():
        raise ValueError("The GT mask is empty, no positive points can be sampled")
    rng = np.random.default_rng(seed)
    region = gt
    if margin > 0:
        eroded = ndi.binary_erosion(gt, iterations=margin)
        # Thin or small masks erode to nothing, the full mask is used then
        if eroded.any():
            region = eroded
    return _sample_region(region, k_points, rng)

def sample_neg_points(gt, k_points, seed, margin):
    """Sampling k points outside the GT mask, at least margin pixels from the boundary when the image allows it"""
    rng = np.random.default_rng(seed + 1_000_003)  # Different stream than the positive points
    region = ~gt
    if margin > 0:
        far = ~ndi.binary_dilation(gt, iterations=margin)
        if far.any():
            region = far
    if not region.any():
        raise ValueError("The GT mask covers the whole image, no negative points can be sampled")
    return _sample_region(region, k_points, rng)

def sample_ref_points(gt, cfg, seed):
    return (sample_pos_points(gt, cfg.n_points, seed, cfg.boundary_margin),
            sample_neg_points(gt, cfg.n_points, seed, cfg.boundary_margin))

## 2.3 Feature extraction and heatmaps

In [ ]:
BACKBONE_HUB = {"dinov2_s14": "dinov2_vits14", "dinov2_b14": "dinov2_vitb14", "dinov2_l14": "dinov2_vitl14", "dinov2_g14": "dinov2_vitg14"}
model_labels = {"dinov2_s14": "Small", "dinov2_b14": "Base", "dinov2_l14": "Large", "dinov2_g14": "Giant"}
backbone_order = ["Small", "Base", "Large", "Giant"]

_loaded_backbone = {}

def get_backbone(name):
    """Loads one backbone at a time. The previously loaded one is released to free GPU memory,
    which matters since Giant alone takes several GB."""
    if name not in _loaded_backbone:
        _loaded_backbone.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model = torch.hub.load(DINOV2_REPO, BACKBONE_HUB[name], trust_repo=True, skip_validation=True)
        _loaded_backbone[name] = model.to(device).eval()
    return _loaded_backbone[name]

def vit_input_size(h, w, px_um, cfg, patch):
    """DINO input size that keeps the aspect ratio. Optionally resamples to a common physical pixel size."""
    if cfg.target_um_per_px is not None:
        new_h = h * px_um[0] / cfg.target_um_per_px
        new_w = w * px_um[1] / cfg.target_um_per_px
        scale = min(1.0, cfg.vit_eval_size / max(new_h, new_w))
        new_h, new_w = new_h * scale, new_w * scale
    else:
        scale = cfg.vit_eval_size / max(h, w)
        new_h, new_w = h * scale, w * scale
    # Rounding to whole patches
    return max(patch, int(round(new_h / patch)) * patch), max(patch, int(round(new_w / patch)) * patch)

@torch.no_grad()
def get_patch_embeddings(model, case, cfg):
    """Grabbing the normalized DINO patch embeddings for one slice. Returns the [H_grid, W_grid, D] grid
    and the original image height and width."""
    img_t = control_img(case["img"])
    patch = int(model.patch_size)
    # Repeating the single channel since RGB is expected
    x = img_t.unsqueeze(0).repeat(1, 3, 1, 1).to(device)
    h, w = x.shape[-2:]
    in_h, in_w = vit_input_size(h, w, case.get("px_um", (1.0, 1.0)), cfg, patch)
    x = F.interpolate(x, (in_h, in_w), mode="bilinear", align_corners=False)
    x = (x - imagenet_mn) / imagenet_std
    # n=k returns the last k blocks, so index 0 is the k-th block from the end
    tokens = model.get_intermediate_layers(x, n=cfg.dino_layer_from_end, reshape=False, return_class_token=False, norm=True)[0]
    patch_embeddings = tokens[0].reshape(in_h // patch, in_w // patch, -1)
    return F.normalize(patch_embeddings, dim=-1), h, w

def prototypes(patch_embeddings, h, w, ref_points):
    """Picks the embeddings of the patches that contain the reference points"""
    h_grid, w_grid = patch_embeddings.shape[:2]
    pts = np.asarray(ref_points)
    y_patch = np.clip((pts[:, 0] / h * h_grid).astype(int), 0, h_grid - 1)
    x_patch = np.clip((pts[:, 1] / w * w_grid).astype(int), 0, w_grid - 1)
    y_patch = torch.as_tensor(y_patch, device=patch_embeddings.device, dtype=torch.long)
    x_patch = torch.as_tensor(x_patch, device=patch_embeddings.device, dtype=torch.long)
    return patch_embeddings[y_patch, x_patch]

@torch.no_grad()
def heatmap_from_protos(patch_embeddings, pos_proto, neg_proto, out_shape, neg_weight):
    """Mean cosine similarity to the positive prototypes minus lambda times that to the negative ones,
    upsampled to the image size. Mean aggregation worked best in earlier tests."""
    H, W, D = patch_embeddings.shape
    feats = patch_embeddings.reshape(-1, D)
    pos_map = feats @ F.normalize(pos_proto, dim=-1).T
    neg_map = feats @ F.normalize(neg_proto, dim=-1).T
    sim_grid = (pos_map.mean(dim=1) - neg_weight * neg_map.mean(dim=1)).reshape(H, W)
    if CFG.score_norm == "robust":
        med = sim_grid.median()
        sim_grid = (sim_grid - med) / ((sim_grid - med).abs().median() * 1.4826 + 1e-6)
    heat = F.interpolate(sim_grid[None, None], size=tuple(out_shape), mode="bilinear", align_corners=False)[0, 0]
    return heat.float().cpu().numpy()

def heatmap_for_case(patch_embeddings, h, w, gt, cfg, seed, return_points=False):
    """Samples reference points from the GT mask and computes the heatmap of the same slice.
    With return_points, also returns the points, which the clicks threshold needs."""
    pos_pts, neg_pts = sample_ref_points(gt, cfg, seed)
    pos_proto = prototypes(patch_embeddings, h, w, pos_pts)
    neg_proto = prototypes(patch_embeddings, h, w, neg_pts)
    heat = heatmap_from_protos(patch_embeddings, pos_proto, neg_proto, (h, w), cfg.neg_weight)
    return (heat, pos_pts, neg_pts) if return_points else heat

@torch.no_grad()
def dino_segment(model, case, cfg, seed):
    """Full DINO pipeline for one slice. The embeddings are computed only once."""
    patch_embeddings, h, w = get_patch_embeddings(model, case, cfg)
    heat, _pp, _nn = heatmap_for_case(patch_embeddings, h, w, case["gt"], cfg, seed, return_points=True)
    pred, thr = threshold_heatmap(heat, cfg, _pp, _nn)
    return heat, pred, thr

## 2.4 Experiment helpers

In [ ]:
@torch.no_grad()
def evaluate_dino(model_name, model, cases, cfg, seeds=None, with_hd95=True):
    """Evaluates the DINO pipeline on the given slices"""
    rows = []
    for i, case in enumerate(cases):
        patch_embeddings, h, w = get_patch_embeddings(model, case, cfg)
        for seed in seeds or cfg.seeds:
            heat, _pp, _nn = heatmap_for_case(patch_embeddings, h, w, case["gt"], cfg, seed, return_points=True)
            pred, thr = threshold_heatmap(heat, cfg, _pp, _nn)
            rows.append({"model": model_name, "file": case["file"], "z": case["z"], "seed": seed, "threshold": thr,
                         **seg_metrics(pred, case["gt"], case["px_um"], with_hd95)})
    return pd.DataFrame(rows)

# Settings that change the embeddings themselves, so the backbone has to be rerun for every value
_EMBEDDING_PARAMS = {"dino_layer_from_end", "vit_eval_size", "target_um_per_px"}

@torch.no_grad()
def sweep(model, cases, cfg, param, values, seeds=None):
    """Evaluates one setting at a time on the given slices, with all other settings taken from cfg.
    Embeddings are reused between values unless the setting changes them."""
    rows = []
    for i, case in enumerate(cases):
        if i % 25 == 0:
            print(f"  {param}: slice {i+1}/{len(cases)}")
        if param not in _EMBEDDING_PARAMS:
            emb = get_patch_embeddings(model, case, cfg)
        for value in values:
            cfg_v = replace(cfg, **{param: value})
            if param in _EMBEDDING_PARAMS:
                emb = get_patch_embeddings(model, case, cfg_v)
            patch_embeddings, h, w = emb
            for seed in seeds or cfg.seeds:
                heat, _pp, _nn = heatmap_for_case(patch_embeddings, h, w, case["gt"], cfg_v, seed, return_points=True)
                pred, thr = threshold_heatmap(heat, cfg_v, _pp, _nn)
                rows.append({"param": param, "value": value, "file": case["file"], "z": case["z"], "seed": seed,
                             "threshold": thr, **seg_metrics(pred, case["gt"], case["px_um"], with_hd95=False)})
    return pd.DataFrame(rows)

def _ci95(x):
    """t-based 95% confidence half-width"""
    x = pd.Series(x).dropna()
    n = len(x)
    return np.nan if n < 2 else float(stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n))

def summarize(df, metric, by=(), cluster="file", unit=("z",)):
    """Two-level summary that respects the correlation in the data.
    1. Averages over seeds for each slice (unit).
    2. Averages over slices for each sample (cluster). The spread over slices is reported as an SD, not a CI,
       since neighbouring slices are not independent.
    3. Averages over samples, with a t-based 95% CI that treats the sample as the independent unit.
    Returns the per-sample table and the overall table."""
    by, unit = list(by), list(unit)
    d = df.copy()
    if not by:
        d["_all"] = "all"
        by = ["_all"]
    per_unit = d.groupby(by + [cluster] + unit, observed=True)[metric].mean().reset_index()
    per_cluster = (per_unit.groupby(by + [cluster], observed=True)[metric]
                   .agg(Mean="mean", SD_slices="std", n_slices="count").reset_index())
    overall = (per_cluster.groupby(by, observed=True)["Mean"]
               .agg(Mean="mean", SD_samples="std", n_samples="count", CI95=_ci95).reset_index())
    per_cluster = per_cluster.rename(columns={"Mean": f"Mean_{metric}"})
    overall = overall.rename(columns={"Mean": f"Mean_{metric}", "SD_samples": f"SD_samples_{metric}", "CI95": f"CI95_{metric}"})
    if by == ["_all"]:
        per_cluster, overall = per_cluster.drop(columns="_all"), overall.drop(columns="_all")
    return per_cluster, overall

def best_value(df_sweep, metric="Dice"):
    """The setting with the highest mean over samples"""
    _, overall = summarize(df_sweep, metric, by=["value"])
    return overall.sort_values(f"Mean_{metric}", ascending=False).iloc[0]["value"]

# 3. Experiments

## 3.1 Dataset creation

File B lacks many annotated slices and is left out. The files are listed in `FILES`. Every file uses image channel 4, except file A, where channel 3 is the right one.

In [ ]:
FILES = {
    "A": {"filename": "10-26-40_6_Blaze_crop2 quantified.ims", "img_channel": 3},
    "C": {"filename": "16-22-39_IC 4_Trap_SOST_col1-_Blaze crop1 segmentation quantified.ims", "img_channel": 4},
    "D": {"filename": "15-15-06_IC 2_Trap_SOST_col1_Blaze crop2 segmentation quantified.ims", "img_channel": 4},
    "E": {"filename": "11-26-23_1TRAP_SOST_NCAM_Blaze crop3 segmentation quantified.ims", "img_channel": 4},
    "F": {"filename": "11-30-50_5TRAP_SOST_NCAM_Blazecrop1 segmentation quantified.ims", "img_channel": 4},
}
file_order = list(FILES)

cases_per_file, tuning_cases_per_file, eval_cases_per_file = {}, {}, {}
for file_name, info in FILES.items():
    print(f"Loading file {file_name}")
    cases = load_file_cases(file_name, info, CFG)
    cases_per_file[file_name] = cases
    # Every tuning_every-th slice, offset by one, is reserved for hyperparameter tuning
    tuning_cases_per_file[file_name] = [c for c in cases if c["slice_idx"] % CFG.tuning_every == 1]
    eval_cases_per_file[file_name] = [c for c in cases if c["slice_idx"] % CFG.tuning_every != 1]
    px = cases[0]["px_um"]
    print(f"  n={len(cases)} (tuning {len(tuning_cases_per_file[file_name])}, evaluation {len(eval_cases_per_file[file_name])}), "
          f"image {cases[0]['img'].shape}, pixel size {px[0]:.3f} x {px[1]:.3f} um")

tuning_cases = [c for f in file_order for c in tuning_cases_per_file[f]]
eval_cases = [c for f in file_order for c in eval_cases_per_file[f]]

## 3.2 Hyperparameter tuning and sensitivity

Settings are chosen on the tuning slices, separately for every backbone, one setting at a time in the order layer, lambda, threshold. The best value of each is kept before the next one is tuned. A layer that suits Giant (40 blocks) can be poor for Small (12 blocks), so sharing Giant's settings would understate the smaller backbones. `CFG` keeps Giant's settings, which the transfer experiments use; `CFG_BY_BACKBONE` holds every backbone's own. The number of reference points and the boundary margin describe how a user would click, so they are only reported as sensitivity analyses, for Giant.

In [ ]:
DINO_KEYS = ("dino_layer_from_end", "neg_weight", "threshold_mode", "top_percent")

def tune_backbone(name, cfg):
    """Tunes layer, lambda and threshold for one backbone on the tuning slices. Returns its settings and the sweeps."""
    model = get_backbone(name)
    res = {}
    res["dino_layer_from_end"] = sweep(model, tuning_cases, cfg, "dino_layer_from_end", (1, 2, 3, 4))
    cfg = replace(cfg, dino_layer_from_end=int(best_value(res["dino_layer_from_end"])))
    res["neg_weight"] = sweep(model, tuning_cases, cfg, "neg_weight", (0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2))
    cfg = replace(cfg, neg_weight=float(best_value(res["neg_weight"])))
    res["top_percent"] = sweep(model, tuning_cases, replace(cfg, threshold_mode="top_percent"), "top_percent", (2, 4, 6, 8, 10, 12, 14, 16, 18, 20))
    cfg = replace(cfg, top_percent=float(best_value(res["top_percent"])))
    # The tuned fixed-percentage threshold against Otsu, which assumes no foreground fraction, and against a threshold
    # calibrated from the reference points themselves ("clicks"), which the boneseg app uses by default
    res["threshold_mode"] = sweep(model, tuning_cases, cfg, "threshold_mode", ("top_percent", "otsu", "clicks"))
    cfg = replace(cfg, threshold_mode=str(best_value(res["threshold_mode"])))
    return cfg, res, model

CFG_BY_BACKBONE, TUNING_BY_BACKBONE = {}, {}
for _name in ("dinov2_s14", "dinov2_b14", "dinov2_l14", "dinov2_g14"):   # Giant last, so it stays loaded for the sensitivity sweeps
    print(f"Tuning {_name}")
    tune_model = None   # drop the previous backbone before the next one loads
    CFG_BY_BACKBONE[_name], TUNING_BY_BACKBONE[_name], tune_model = tune_backbone(_name, CFG)
    print({k: getattr(CFG_BY_BACKBONE[_name], k) for k in DINO_KEYS})
CFG = CFG_BY_BACKBONE["dinov2_g14"]
tuning_results = dict(TUNING_BY_BACKBONE["dinov2_g14"])

# Sensitivity analyses only, CFG is not changed
tuning_results["n_points"] = sweep(tune_model, tuning_cases, CFG, "n_points", (1, 3, 5, 10, 15, 25, 50), seeds=(0, 1, 2, 3, 4))
tuning_results["boundary_margin"] = sweep(tune_model, tuning_cases, CFG, "boundary_margin", (0, 5, 10, 20))

for param, df_sweep in tuning_results.items():
    _, overall = summarize(df_sweep, "Dice", by=["value"])
    print(f"\n{param}")
    display(overall.round(4))

import pandas as _pd
save_table(_pd.concat(tuning_results.values(), ignore_index=True), "tuning_sweeps")
for _param, _df in tuning_results.items():
    save_table(summarize(_df, "Dice", by=["value"])[1], f"tuning_{_param}")
# Every backbone's own settings and sweeps
save_table(_pd.DataFrame([{"backbone": n, **{k: getattr(c, k) for k in DINO_KEYS}} for n, c in CFG_BY_BACKBONE.items()]), "settings_by_backbone")
save_table(_pd.concat([summarize(df, "Dice", by=["value"])[1].assign(backbone=n, param=param)
                       for n, res in TUNING_BY_BACKBONE.items() for param, df in res.items()], ignore_index=True), "tuning_by_backbone")
# Giant is not needed again until the transfer experiment; dropping this reference lets get_backbone free it
del tune_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Giant's settings, used from here on except where a backbone's own are noted:")
print({k: getattr(CFG, k) for k in ("dino_layer_from_end", "neg_weight", "threshold_mode", "top_percent", "n_points", "boundary_margin")})

## 3.3 DINO backbone experiments

Every backbone is evaluated on the evaluation slices with its own settings, chosen on the tuning slices. HD95 is reported in micrometres.

In [ ]:
backbone_results = []
for model_name in BACKBONE_HUB:
    print("Evaluating:", model_name)
    # Each backbone with its own tuned settings
    backbone_results.append(evaluate_dino(model_name, get_backbone(model_name), eval_cases, CFG_BY_BACKBONE[model_name]))
df_dino_backbone = pd.concat(backbone_results, ignore_index=True)
df_dino_backbone["Backbone"] = pd.Categorical(df_dino_backbone["model"].map(model_labels), categories=backbone_order, ordered=True)

summary_dino_backbone = None
for metric in ("Dice", "IoU", "HD95_um"):
    _, overall = summarize(df_dino_backbone, metric, by=["Backbone"])
    summary_dino_backbone = overall if summary_dino_backbone is None else summary_dino_backbone.merge(
        overall.drop(columns="n_samples"), on="Backbone")
print("Mean over samples, with 95% CI across the five samples")
display(summary_dino_backbone.round(4))
save_table(df_dino_backbone.drop(columns=["Backbone"]), "dino_backbone")
save_table(summary_dino_backbone, "dino_backbone_summary")

per_file_backbone, _ = summarize(df_dino_backbone, "Dice", by=["Backbone"])
print("Per sample")
display(per_file_backbone.round(4))
n_failed = df_dino_backbone["HD95_um"].isna().sum()
if n_failed:
    print(f"{n_failed} predictions were empty, their HD95 is undefined and left out of the HD95 means")

## 3.4 Robustness and generalization

The transfer experiments use prototype embeddings from a reference slice to segment the same slice, other slices from the same sample and slices from different samples. The 20 target slices per sample are evenly spaced over the evaluation set, and the 5 reference slices are evenly spaced among the targets, so they are not neighbours.

In [ ]:
def case_key(case):
    """Lets slices be looked up by sample and z-position, to reuse precomputed embeddings"""
    return (case["file"], case["z"])

def transfer_type(ref_case, target_case):
    """Classifies the type of the transfer"""
    if case_key(ref_case) == case_key(target_case):
        return "same_slice"
    if ref_case["file"] == target_case["file"]:
        return "same_file"
    return "other_file"

@torch.no_grad()
def eval_transfers(model, cfg, files, n_ref_per_file, n_target_per_file, seeds):
    """Evaluating the prototype transfer between samples and slices"""
    target_cases, ref_cases = [], []
    for f in files:
        targets = evenly_spaced(eval_cases_per_file[f], n_target_per_file)
        target_cases.extend(targets)
        ref_cases.extend(evenly_spaced(targets, n_ref_per_file))
    cache = {}
    for idx, case in enumerate(target_cases):
        print(f"Precomputing {idx+1}/{len(target_cases)}: file={case['file']}, z={case['z']}")
        cache[case_key(case)] = get_patch_embeddings(model, case, cfg)
    rows = []
    for seed in seeds:
        print(f"Seed: {seed}")
        for ref_case in ref_cases:
            patch_embeddings, h, w = cache[case_key(ref_case)]
            pos_pts, neg_pts = sample_ref_points(ref_case["gt"], cfg, seed)
            pos_proto = prototypes(patch_embeddings, h, w, pos_pts)
            neg_proto = prototypes(patch_embeddings, h, w, neg_pts)
            # The clicks threshold is calibrated on the reference slice and carried to every target
            ref_heat = heatmap_from_protos(patch_embeddings, pos_proto, neg_proto, (h, w), cfg.neg_weight)
            _, ref_thr = threshold_heatmap(ref_heat, cfg, pos_pts, neg_pts)
            for target_case in target_cases:
                target_emb, th, tw = cache[case_key(target_case)]
                heat = heatmap_from_protos(target_emb, pos_proto, neg_proto, (th, tw), cfg.neg_weight)
                if cfg.threshold_mode == "clicks":
                    pred, thr = heat >= ref_thr, ref_thr
                else:
                    pred, thr = threshold_heatmap(heat, cfg)
                rows.append({"seed": seed, "ref_file": ref_case["file"], "ref_z": ref_case["z"],
                             "target_file": target_case["file"], "target_z": target_case["z"],
                             "case_type": transfer_type(ref_case, target_case), "threshold": thr,
                             **seg_metrics(pred, target_case["gt"], target_case["px_um"])})
    return pd.DataFrame(rows)

df_ref_transfer = eval_transfers(get_backbone("dinov2_g14"), CFG, file_order, n_ref_per_file=5, n_target_per_file=20, seeds=CFG.seeds)
transfer_labels = {"same_slice": "Same slice", "same_file": "Same sample", "other_file": "Different sample"}
df_ref_transfer["Transfer"] = pd.Categorical(df_ref_transfer["case_type"].map(transfer_labels), categories=list(transfer_labels.values()), ordered=True)

pairwise_transfer_table = (df_ref_transfer.groupby(["ref_file", "target_file"], observed=True)
                           .agg(Mean_Dice=("Dice", "mean"), Mean_IoU=("IoU", "mean"), N=("Dice", "count")).reset_index())
display(pairwise_transfer_table.round(4))

# The target sample is the independent unit, each observation is one reference and target slice pair
_, summary_transfer = summarize(df_ref_transfer, "Dice", by=["Transfer"], cluster="target_file", unit=("ref_file", "ref_z", "target_z"))
display(summary_transfer.round(4))
save_table(df_ref_transfer.drop(columns=["Transfer"]), "transfer")
save_table(pairwise_transfer_table, "transfer_pairwise")
save_table(summary_transfer, "transfer_summary")

# 4. U-Net addition

A U-Net is trained on 1) the raw image, 2) the DINO heatmap, 3) the raw image and the DINO heatmap or 4) the raw image, the DINO heatmap and the DINO segmentation mask. The DINO inputs come from the Small backbone with the settings tuned in section 3.2.

Leave-one-sample-out cross-validation is used. For each held-out sample, the U-Net trains on slices from the four other samples. Validation slices from those four samples, interleaved with the training slices, pick the best epoch and the probability threshold. The held-out sample is only used for the final comparison with DINO, on slices from its evaluation set. A linear probe on the DINO features (section 4.2b) runs in the same loop as a lighter supervised baseline.

In [ ]:
class UnetPrecomp(Dataset):
    """Combines the precomputed inputs and GT masks, with optional flip and rotation augmentation"""
    def __init__(self, items, channels, augment=False):
        self.items = items
        self.channels = list(channels)
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x = self.items[idx]["x"][self.channels].float()
        y = self.items[idx]["y"].float()
        if self.augment:
            k = random.randint(0, 3)
            x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (1, 2))
            if random.random() < 0.5:
                x, y = x.flip(-1), y.flip(-1)
        return x.contiguous(), y.contiguous()

class DoubleConv(nn.Module):
    """Convolution block used in the encoder and decoder"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.double_conv(x)

class DownBlock(nn.Module):
    """Encoder block: returns the downsampled features and the skip connection features"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.double_conv = DoubleConv(in_c, out_c)
        self.down_sample = nn.MaxPool2d(2)

    def forward(self, x):
        skip_out = self.double_conv(x)
        return self.down_sample(skip_out), skip_out

class UpBlock(nn.Module):
    """Decoder block: upsampling and merging with the skip connection features"""
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up_sample = nn.ConvTranspose2d(in_c, out_c, 2, 2)
        self.double_conv = DoubleConv(out_c + skip_c, out_c)

    def forward(self, down_input, skip_input):
        x = self.up_sample(down_input)
        return self.double_conv(torch.cat([x, skip_input], dim=1))

class UNet(nn.Module):
    """U-Net with len(features)-1 downsampling steps, outputs logits.
    The default 4 levels give a larger receptive field than the earlier 2-level version, which matters at 960 px."""
    def __init__(self, in_c=2, out_c=1, features=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        c = in_c
        for f in features[:-1]:
            self.downs.append(DownBlock(c, f))
            c = f
        self.bottleneck = DoubleConv(c, features[-1])
        self.ups = nn.ModuleList()
        c = features[-1]
        for f in reversed(features[:-1]):
            self.ups.append(UpBlock(c, f, f))
            c = f
        self.conv_last = nn.Conv2d(c, out_c, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x, skip = down(x)
            skips.append(skip)
        x = self.bottleneck(x)
        for up, skip in zip(self.ups, reversed(skips)):
            x = up(x, skip)
        return self.conv_last(x)

## 4.1 Loss and training functions

In [ ]:
def soft_dice_loss(logits, targ, eps=1e-6):
    """Soft dice loss, as used in the combo loss by Taghanaki et al."""
    probs = torch.sigmoid(logits)
    inter = (probs * targ).sum(dim=(1, 2, 3))
    denom = probs.sum(dim=(1, 2, 3)) + targ.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps)).mean()

def combo_loss(logits, targ):
    """Combines BCE and dice loss, weighting them 50/50"""
    return F.binary_cross_entropy_with_logits(logits, targ) / 2 + soft_dice_loss(logits, targ) / 2

# Mixed precision on CUDA roughly halves the activation memory of the 960 px U-Nets, which ran out of memory
# on a 15 GB T4 in full precision. On other devices everything stays in full precision.
USE_AMP = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def one_epoch(loader, model, optimizer):
    """Trains one epoch and returns the mean training loss"""
    model.train()
    losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        loss = combo_loss(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    return float(np.mean(losses))

@torch.no_grad()
def thr_sweep(loader, model, thresholds):
    """Evaluates the probability thresholds on a loader, sorted by mean dice"""
    model.eval()
    probs_all, targs_all = [], []
    for x, y in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(device))
        probs_all.extend(torch.sigmoid(logits.float()).squeeze(1).cpu().numpy())
        targs_all.extend(y.squeeze(1).numpy().astype(bool))
    rows = []
    for thr in thresholds:
        dices = [dice(p >= thr, g) for p, g in zip(probs_all, targs_all)]
        ious = [iou(p >= thr, g) for p, g in zip(probs_all, targs_all)]
        rows.append({"threshold": thr, "Dice_mean": np.mean(dices), "Dice_std": np.std(dices), "IoU_mean": np.mean(ious), "IoU_std": np.std(ious)})
    return pd.DataFrame(rows).sort_values("Dice_mean", ascending=False).reset_index(drop=True)

def run_unet(train_loader, val_loader, in_c, run_name, cfg, seed):
    """Trains the U-Net and keeps the epoch with the best validation dice.
    The validation loader must not contain the test slices."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    seeding(seed)
    model = UNet(in_c=in_c, out_c=1, features=cfg.unet_features).to(device)
    optimizer = torch.optim.Adam(model.parameters(), cfg.unet_lr)
    best_val_dice, best_epoch, best_thr, no_improve = -1.0, -1, None, 0
    history = []
    path = CHECKPOINT_DIR / f"best_{run_name}.pt"

    for epoch in range(cfg.unet_epochs):
        train_loss = one_epoch(train_loader, model, optimizer)
        best_row = thr_sweep(val_loader, model, cfg.unet_thresholds).iloc[0]
        val_dice, val_iou, thr = float(best_row["Dice_mean"]), float(best_row["IoU_mean"]), float(best_row["threshold"])
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_dice": val_dice, "val_iou": val_iou, "best_thr": thr})
        if val_dice > best_val_dice:
            best_val_dice, best_epoch, best_thr, no_improve = val_dice, epoch + 1, thr, 0
            torch.save(model.state_dict(), path)
        else:
            no_improve += 1
        print(f"Epoch {epoch+1:02d}: train_loss={train_loss:.4f}, val_dice={val_dice:.4f}, val_iou={val_iou:.4f}, best_thr={thr:.2f}")
        if no_improve >= cfg.unet_patience:
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    print(f"Best epoch {best_epoch}, val dice {best_val_dice:.4f}, threshold {best_thr}, checkpoint {path}")
    return {"run_name": run_name, "model": model, "best_val_dice": best_val_dice, "best_thr": best_thr,
            "best_epoch": best_epoch, "history_df": pd.DataFrame(history), "path": str(path)}

## 4.2 Input creation and data loading

In [ ]:
# All variants are channel subsets of the same three-channel stack: raw image, DINO heatmap, DINO mask
VARIANT_CHANNELS = {"raw": [0], "heat": [1], "rawheat": [0, 1], "rawheatmask": [0, 1, 2]}

def minmax(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / max(float(a.max() - a.min()), 1e-8)

def build_unet_stack(raw_img, heat, pred_dino):
    """Stacks the normalized raw image, the normalized heatmap and the DINO mask into [3, H, W]"""
    return torch.from_numpy(np.stack([minmax(raw_img), minmax(heat), np.asarray(pred_dino, np.float32)], axis=0))

def to_unet_size(x, size, mode):
    if mode == "nearest":
        return F.interpolate(x[None], (size, size), mode="nearest")[0]
    return F.interpolate(x[None], (size, size), mode=mode, align_corners=False)[0]

@torch.no_grad()
def dino_outputs(model_dino, cases, cfg, seed):
    """Runs DINO once per slice and returns the raw image, heatmap and mask at full resolution"""
    out = []
    for case in cases:
        heat, pred_dino, _ = dino_segment(model_dino, case, cfg, seed)
        out.append({"case": case, "raw": np.asarray(case["img"], np.float32), "heat": heat, "pred_dino": pred_dino})
    return out

def precompute_unet_items(model_dino, cases, cfg, seed):
    """U-Net inputs and GT masks at the U-Net input size. Computed once and shared by all variants and folds."""
    items = []
    for d in dino_outputs(model_dino, cases, cfg, seed):
        x = to_unet_size(build_unet_stack(d["raw"], d["heat"], d["pred_dino"]), cfg.unet_size, "bilinear")
        y = to_unet_size(torch.from_numpy(d["case"]["gt"].astype(np.float32))[None], cfg.unet_size, "nearest")
        items.append({"x": x, "y": y, "file": d["case"]["file"], "z": d["case"]["z"]})
    return items

@torch.no_grad()
def predict_unet(unet_model, d, variant, threshold, cfg):
    """Predicts a full-resolution mask for one slice from its DINO outputs"""
    org_h, org_w = d["raw"].shape
    x = build_unet_stack(d["raw"], d["heat"], d["pred_dino"])[VARIANT_CHANNELS[variant]]
    x = to_unet_size(x, cfg.unet_size, "bilinear")[None].to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        logits = unet_model(x)
    probs = torch.sigmoid(logits.float())
    probs = F.interpolate(probs, (org_h, org_w), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    return probs, probs >= threshold

def dino_vs_unet(unet_model, test_outputs, variant, threshold, cfg):
    """Compares DINO and U-Net per test slice"""
    rows = []
    for d in test_outputs:
        case = d["case"]
        _, pred_unet = predict_unet(unet_model, d, variant, threshold, cfg)
        m_dino = seg_metrics(d["pred_dino"], case["gt"], case["px_um"])
        m_unet = seg_metrics(pred_unet, case["gt"], case["px_um"])
        rows.append({"file": case["file"], "z": case["z"], **{f"{k}_DINO": v for k, v in m_dino.items()}, **{f"{k}_UNET": v for k, v in m_unet.items()}})
    return pd.DataFrame(rows)

## 4.2b A linear probe on the DINO features

A much lighter supervised baseline than the U-Net: a linear or small two-layer classifier on the frozen DINO patch features, fitted to the share of each patch covered by the expert mask, optionally with the mean features of the 5 x 5 neighbouring patches. It trains in seconds and is the model behind the boneseg app's "Learned model". It uses the same training, validation and test slices as the U-Nets, and its settings are picked on the validation slices.

In [ ]:
def probe_features(emb, context):
    """[N, D] patch features, optionally joined with the mean of the context x context neighbourhood"""
    g = emb.float().cpu()
    if context > 1:
        t = g.permute(2, 0, 1)[None]
        ctx = F.avg_pool2d(t, context, stride=1, padding=context // 2, count_include_pad=False)[0].permute(1, 2, 0)
        g = torch.cat([g, ctx], -1)
    return g.reshape(-1, g.shape[-1])

def fit_probe(samples, kind, context, epochs=150, seed=0):
    """samples: [(patch_embeddings [H, W, D], gt mask)]. Patches are weighted so both classes count equally."""
    torch.manual_seed(seed)
    X = torch.cat([probe_features(e, context) for e, _ in samples])
    y = torch.cat([F.adaptive_avg_pool2d(torch.from_numpy(m.astype(np.float32))[None, None], e.shape[:2])[0, 0].reshape(-1) for e, m in samples])
    share = float(y.mean().clamp(1e-3, 1 - 1e-3))
    w = torch.where(y > 0.5, 0.5 / share, 0.5 / (1 - share))
    model = nn.Linear(X.shape[1], 1) if kind == "linear" else nn.Sequential(nn.Linear(X.shape[1], 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 1))
    opt = torch.optim.Adam(model.parameters(), lr=1e-2 if kind == "linear" else 3e-3, weight_decay=1e-3)
    for _ in range(epochs):
        opt.zero_grad()
        loss = (F.binary_cross_entropy_with_logits(model(X).squeeze(1), y, reduction="none") * w).mean()
        loss.backward()
        opt.step()
    return model.eval()

@torch.no_grad()
def probe_predict(model, emb, context, shape):
    logits = model(probe_features(emb, context)).reshape(emb.shape[:2])
    prob = F.interpolate(torch.sigmoid(logits)[None, None], size=tuple(shape), mode="bilinear", align_corners=False)[0, 0]
    return prob.numpy() >= 0.5

def probe_embeddings(cases):
    return {case_key(c): (get_patch_embeddings(unet_dino, c, CFG_UNET_DINO)[0].cpu(), c["gt"]) for c in cases}

## 4.3 Leave-one-sample-out runs

In [ ]:
# Free the larger backbones and their cached features before training, the U-Nets need the GPU memory
_loaded_backbone.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
unet_dino = get_backbone(CFG.unet_backbone)
# DINO inputs for the U-Nets and the probe use the U-Net backbone's own tuned settings, U-Net settings stay in CFG
CFG_UNET_DINO = replace(CFG, **{k: getattr(CFG_BY_BACKBONE[CFG.unet_backbone], k) for k in DINO_KEYS})
print("DINO settings for the U-Net inputs:", {k: getattr(CFG_UNET_DINO, k) for k in DINO_KEYS})
n_unet = CFG.unet_slices_per_file
# Interleaved training and validation slices within each sample, and test slices from the evaluation set
unet_train_pool = {f: evenly_spaced([c for c in cases_per_file[f] if c["slice_idx"] % 4 == 0], n_unet) for f in file_order}
unet_val_pool = {f: evenly_spaced([c for c in cases_per_file[f] if c["slice_idx"] % 4 == 2], n_unet) for f in file_order}
unet_test_cases = {f: evenly_spaced(eval_cases_per_file[f], n_unet) for f in file_order}

all_compare, saved_results, histories = [], {}, []
for seed in CFG.unet_seeds:
    print(f"Seed {seed}: precomputing DINO inputs")
    train_items = {f: precompute_unet_items(unet_dino, unet_train_pool[f], CFG_UNET_DINO, seed) for f in file_order}
    val_items = {f: precompute_unet_items(unet_dino, unet_val_pool[f], CFG_UNET_DINO, seed) for f in file_order}
    probe_train = probe_embeddings([c for f in file_order for c in unet_train_pool[f]])
    probe_val = probe_embeddings([c for f in file_order for c in unet_val_pool[f]])
    for test_file in file_order:
        print(f"\nHeld-out sample: {test_file}")
        train_files = [f for f in file_order if f != test_file]
        test_outputs = dino_outputs(unet_dino, unet_test_cases[test_file], CFG_UNET_DINO, seed)

        # The linear probe: settings picked on the validation slices of the training samples
        tr = [v for k, v in probe_train.items() if k[0] in train_files]
        va = [v for k, v in probe_val.items() if k[0] in train_files]
        best = None
        for kind in ("linear", "mlp"):
            for context in (1, 5):
                m = fit_probe(tr, kind, context, seed=seed)
                d = np.mean([dice(probe_predict(m, e, context, g.shape), g) for e, g in va])
                if best is None or d > best[0]:
                    best = (d, kind, context, m)
        print(f"Probe: {best[1]}, context {best[2]}, validation Dice {best[0]:.3f}")
        rows = []
        for d in test_outputs:
            case = d["case"]
            emb = get_patch_embeddings(unet_dino, case, CFG_UNET_DINO)[0].cpu()
            pred = probe_predict(best[3], emb, best[2], case["gt"].shape)
            m_dino = seg_metrics(d["pred_dino"], case["gt"], case["px_um"])
            m_probe = seg_metrics(pred, case["gt"], case["px_um"])
            rows.append({"file": case["file"], "z": case["z"], **{f"{k}_DINO": v for k, v in m_dino.items()}, **{f"{k}_UNET": v for k, v in m_probe.items()}})
        df = pd.DataFrame(rows)
        df["seed"], df["variant"], df["best_thr"], df["best_val_dice"], df["best_epoch"] = seed, "probe", 0.5, best[0], np.nan
        all_compare.append(df)
        for variant, channels in VARIANT_CHANNELS.items():
            print(f"\nVariant: {variant}, training on {train_files}")
            train_loader = DataLoader(UnetPrecomp([it for f in train_files for it in train_items[f]], channels, augment=True),
                                      batch_size=CFG.unet_batch_size, shuffle=True)
            val_loader = DataLoader(UnetPrecomp([it for f in train_files for it in val_items[f]], channels),
                                    batch_size=CFG.unet_batch_size, shuffle=False)
            res = run_unet(train_loader, val_loader, len(channels), f"unet_{variant}_{test_file}_seed{seed}", CFG, seed)
            df = dino_vs_unet(res["model"], test_outputs, variant, res["best_thr"], CFG)
            df["seed"], df["variant"] = seed, variant
            df["best_thr"], df["best_val_dice"], df["best_epoch"] = res["best_thr"], res["best_val_dice"], res["best_epoch"]
            all_compare.append(df)
            histories.append(res["history_df"].assign(test_file=test_file, variant=variant, seed=seed))
            # Kept on the CPU to save GPU memory, moved back when needed for the plots
            saved_results[(test_file, seed, variant)] = {"model": res["model"].cpu(), "best_thr": res["best_thr"]}
        del test_outputs
    del train_items, val_items, probe_train, probe_val
    gc.collect()

df_compare_models = pd.concat(all_compare, ignore_index=True)
df_compare_models["Dice_gain"] = df_compare_models["Dice_UNET"] - df_compare_models["Dice_DINO"]
df_unet_history = pd.concat(histories, ignore_index=True)
save_table(df_compare_models, "unet_vs_dino")
save_table(df_unet_history, "unet_training_history")
display(df_compare_models.round(4))

In [ ]:
variant_labels = {"raw": "Raw", "heat": "Heatmap", "rawheat": "Raw + Heatmap", "rawheatmask": "Raw + Heatmap + Mask",
                  "probe": "Linear probe on DINO (no U-Net)"}
variant_order = list(VARIANT_CHANNELS) + ["probe"]
df_compare_models["Variant"] = pd.Categorical(df_compare_models["variant"].map(variant_labels), categories=[variant_labels[v] for v in variant_order], ordered=True)

summary_main = None
for metric in ("Dice_DINO", "Dice_UNET", "Dice_gain", "HD95_um_DINO", "HD95_um_UNET"):
    _, overall = summarize(df_compare_models, metric, by=["Variant"])
    summary_main = overall if summary_main is None else summary_main.merge(overall.drop(columns="n_samples"), on="Variant")
print("Test results on the held-out samples, mean over samples with 95% CI across samples")
display(summary_main.round(4))

per_file_gain, _ = summarize(df_compare_models, "Dice_gain", by=["Variant"])
print("Dice gain per held-out sample")
display(per_file_gain.round(4))

# 5. Plots

## 5.1 Segmentation performance for each backbone

Mean dice per sample for each backbone, with error bars showing the SD across slices. The black line is the mean over samples, with a 95% confidence interval across the five samples.

In [ ]:
per_file, overall = summarize(df_dino_backbone, "Dice", by=["Backbone"])
x_positions = {b: i for i, b in enumerate(backbone_order)}

fig, ax = plt.subplots(figsize=(6, 4))
for file_name, sub in per_file.groupby("file"):
    sub = sub.sort_values("Backbone")
    x = [x_positions[str(b)] for b in sub["Backbone"]]
    ax.errorbar(x, sub["Mean_Dice"], yerr=sub["SD_slices"], marker="o", ms=4, capsize=3, linewidth=0.8, alpha=0.75, label=f"Sample {file_name}")
overall = overall.sort_values("Backbone")
ax.errorbar([x_positions[str(b)] for b in overall["Backbone"]], overall["Mean_Dice"], yerr=overall["CI95_Dice"],
            marker="o", markersize=4, linewidth=1.2, capsize=5, color="black", label="Mean over samples")
ax.set_xticks(range(len(backbone_order)))
ax.set_xticklabels(backbone_order)
ax.set_xlabel("Backbone")
ax.set_ylabel("Mean dice")
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5.2 Tuning and sensitivity

Mean dice on the tuning slices as a function of each setting, with a 95% confidence interval across samples. The dashed line marks the value used in the rest of the notebook.

In [ ]:
plt.rcParams.update({"font.size": 12, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11})
sweep_labels = {"dino_layer_from_end": "Block counted from the end", "neg_weight": "Negative weight (lambda)",
                "top_percent": "Top-percent threshold", "n_points": "Number of reference points",
                "boundary_margin": "Boundary margin (px)"}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (param, label) in zip(axes.ravel(), sweep_labels.items()):
    _, overall = summarize(tuning_results[param], "Dice", by=["value"])
    overall = overall.sort_values("value")
    ax.errorbar(overall["value"], overall["Mean_Dice"], yerr=overall["CI95_Dice"], marker="o", capsize=4, linewidth=1.8)
    ax.axvline(getattr(CFG, param), color="black", linestyle="--", linewidth=1, label="Used setting")
    ax.set_xlabel(label)
    ax.set_ylabel("Mean dice")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False)

ax = axes.ravel()[-1]
_, mode_summary = summarize(tuning_results["threshold_mode"], "Dice", by=["value"])
ax.bar(mode_summary["value"].astype(str), mode_summary["Mean_Dice"], yerr=mode_summary["CI95_Dice"], capsize=6, color=["tab:blue", "tab:orange"])
ax.set_xlabel("Threshold mode")
ax.set_ylabel("Mean dice")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 5.3 Transfer matrix

The mean dice scores for the different reference and target samples in the transfer experiments.

In [ ]:
dice_matrix = df_ref_transfer.groupby(["ref_file", "target_file"])["Dice"].mean().unstack().loc[file_order, file_order]
plt.figure(figsize=(7, 6))
ax = sns.heatmap(dice_matrix, annot=True, fmt=".3f", cmap="mako", vmin=0.2, vmax=0.8, square=True, linewidths=0.75,
                 annot_kws={"fontsize": 11}, cbar_kws={"label": "Mean dice score", "shrink": 0.85})
ax.set_xlabel("Target sample", fontsize=12)
ax.set_ylabel("Reference sample", fontsize=12)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

## 5.4 Qualitative results

The worst, median and best dice cases for each sample with the Giant backbone.

In [ ]:
@torch.no_grad()
def qualitative_res(df_results, model, cfg, files):
    fig, axes = plt.subplots(len(files), 3, figsize=(10, 2.8 * len(files)))
    col_titles = ["Worst", "Median", "Best"]
    lookup = {case_key(c): c for f in files for c in cases_per_file[f]}
    for row_idx, file in enumerate(files):
        df_file = df_results[df_results["file"] == file].sort_values("Dice").reset_index(drop=True)
        selected_rows = [df_file.iloc[0], df_file.iloc[len(df_file) // 2], df_file.iloc[-1]]
        for col_idx, row in enumerate(selected_rows):
            case = lookup[(file, int(row["z"]))]
            _, pred, _ = dino_segment(model, case, cfg, int(row["seed"]))
            gt = case["gt"]
            ax = axes[row_idx, col_idx]
            ax.imshow(np.asarray(case["img"], np.float32), cmap="gray")
            ax.contour(pred.astype(float), levels=[0.5], colors="dodgerblue", linewidths=1.3)
            ax.contour(gt.astype(float), levels=[0.5], colors="crimson", linewidths=1.3)
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=12)
            ax.text(0.02, 0.95, f"Sample {file}\nz={case['z']}\nDice={dice(pred, gt):.3f}", transform=ax.transAxes, fontsize=9,
                    va="top", ha="left", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
            ax.axis("off")
    handles = [plt.Line2D([0], [0], color="dodgerblue", lw=2, label="Prediction"), plt.Line2D([0], [0], color="crimson", lw=2, label="Ground truth")]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05)
    plt.show()

qualitative_res(df_dino_backbone[df_dino_backbone["model"] == "dinov2_g14"], get_backbone("dinov2_g14"), CFG, file_order)

## 5.5 Qualitative U-Net results

The lowest, median and highest dice gain of the U-Net over DINO for each held-out sample. Both DINO and the U-Net inputs use the Small backbone, the same one the U-Nets were trained with. Small is used because it combines good segmentation performance with low computational cost.

In [ ]:
@torch.no_grad()
def qualitative_unet(df_compare, saved_results, model_dino, cfg, files, variant):
    fig, axes = plt.subplots(len(files), 3, figsize=(10, 2.8 * len(files)))
    col_titles = ["Lowest gain", "Median gain", "Highest gain"]
    lookup = {case_key(c): c for f in files for c in cases_per_file[f]}
    for row_idx, file in enumerate(files):
        df_file = df_compare[(df_compare["file"] == file) & (df_compare["variant"] == variant)].sort_values("Dice_gain").reset_index(drop=True)
        selected_rows = [df_file.iloc[0], df_file.iloc[len(df_file) // 2], df_file.iloc[-1]]
        for col_idx, row in enumerate(selected_rows):
            seed = int(row["seed"])
            case = lookup[(file, int(row["z"]))]
            d = dino_outputs(model_dino, [case], cfg, seed)[0]
            model_info = saved_results[(file, seed, variant)]
            unet = model_info["model"].to(device)
            _, pred_unet = predict_unet(unet, d, variant, model_info["best_thr"], cfg)
            model_info["model"] = unet.cpu()
            gt = case["gt"]
            dice_dino, dice_unet = dice(d["pred_dino"], gt), dice(pred_unet, gt)
            ax = axes[row_idx, col_idx]
            ax.imshow(d["raw"], cmap="gray")
            ax.contour(d["pred_dino"].astype(float), levels=[0.5], colors="dodgerblue", linewidths=1.2, linestyles="--")
            ax.contour(pred_unet.astype(float), levels=[0.5], colors="lime", linewidths=1.4)
            ax.contour(gt.astype(float), levels=[0.5], colors="crimson", linewidths=1.3)
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=12)
            ax.text(0.02, 0.96, f"Sample {file}, z={case['z']}\nDINO={dice_dino:.3f}\nU-Net={dice_unet:.3f}\nΔ={dice_unet - dice_dino:+.3f}",
                    transform=ax.transAxes, fontsize=8, va="top", ha="left", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
            ax.axis("off")
    handles = [plt.Line2D([0], [0], color="dodgerblue", lw=2, linestyle="--", label="DINO"),
               plt.Line2D([0], [0], color="lime", lw=2, label="U-Net refinement"),
               plt.Line2D([0], [0], color="crimson", lw=2, label="Ground truth")]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05)
    plt.show()

qualitative_unet(df_compare_models, saved_results, get_backbone(CFG.unet_backbone), CFG_UNET_DINO, file_order, "rawheat")

## 5.6 U-Net gain

The U-Net gain over DINO Small for each held-out sample, computed as the U-Net dice minus the DINO dice. Error bars show the SD across slices. With only one held-out sample per bar, a confidence interval across samples is only meaningful for the overall means, which are in the table in section 4.3.

In [ ]:
df_gain = df_compare_models.groupby(["Variant", "file", "z"], observed=True)["Dice_gain"].mean().reset_index()
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df_gain, x="file", y="Dice_gain", hue="Variant", order=file_order,
                 hue_order=[variant_labels[v] for v in variant_order], errorbar="sd", capsize=0.08, err_kws={"linewidth": 1})
ax.axhline(0, color="black")
ax.set_xlabel("Held-out sample")
ax.set_ylabel("Delta dice")
ax.legend(title="U-Net input", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
sns.despine()
plt.tight_layout()
plt.show()

# 6. Save results

The result tables are written as CSV files to the output folder, next to the U-Net checkpoints.

In [ ]:
results_dir = OUTPUT_DIR / "results"
results_dir.mkdir(exist_ok=True)
pd.concat(tuning_results.values(), ignore_index=True).to_csv(results_dir / "tuning_sweeps.csv", index=False)
df_dino_backbone.to_csv(results_dir / "dino_backbone.csv", index=False)
summary_dino_backbone.to_csv(results_dir / "dino_backbone_summary.csv", index=False)
df_ref_transfer.to_csv(results_dir / "transfer.csv", index=False)
summary_transfer.to_csv(results_dir / "transfer_summary.csv", index=False)
df_compare_models.to_csv(results_dir / "unet_vs_dino.csv", index=False)
summary_main.to_csv(results_dir / "unet_vs_dino_summary.csv", index=False)
df_unet_history.to_csv(results_dir / "unet_training_history.csv", index=False)
pd.Series({k: str(v) for k, v in vars(CFG).items()}).to_csv(results_dir / "config_used.csv", header=["value"])
print("Saved results to", results_dir)